## Research Agent
#### Langgraph, GPT-4o, RAG, PineCone, ArXiv and Google SerpAPI

### Extracting Data from ArXiv into a Pandas DataFrame and Saving it as JSON

In [1]:
# pip install -r requirements.txt -q

In [2]:
import requests
import pandas as pd
import json
import xml.etree.ElementTree as ET

# Namespace for ArXiv's Atom-based XML format.
ARXIV_NAMESPACE = '{http://www.w3.org/2005/Atom}'

def extract_from_arxiv(search_query='cat:cs.AI', max_results=100, json_file_path='files/arxiv_dataset.json'):
    """
    Fetches papers from the ArXiv API based on a search query, saves them as JSON, 
    and returns a pandas DataFrame.

    Args:
        search_query (str): The search query for ArXiv (default is 'cat:cs.AI').
        max_results (int): The maximum number of results to retrieve (default is 100).
        json_file_path (str): File path where JSON data will be saved.

    Returns:
        pd.DataFrame: DataFrame containing the extracted paper information.
    """
    
    # Construct the URL for the API request.
    url = f'http://export.arxiv.org/api/query?search_query={search_query}&max_results={max_results}'
    
    # Send a GET request to the ArXiv API.
    response = requests.get(url)
    
    # Parse the XML response.
    root = ET.fromstring(response.content)
    
    papers = []
    
    # Loop through each "entry" in the XML, representing a single paper.
    for entry in root.findall(f'{ARXIV_NAMESPACE}entry'):
        title = entry.find(f'{ARXIV_NAMESPACE}title').text.strip()
        summary = entry.find(f'{ARXIV_NAMESPACE}summary').text.strip()

        # Get the authors of the paper.
        author_elements = entry.findall(f'{ARXIV_NAMESPACE}author')
        authors = [author.find(f'{ARXIV_NAMESPACE}name').text for author in author_elements]

        # Get the paper's URL.
        paper_url = entry.find(f'{ARXIV_NAMESPACE}id').text
        arxiv_id = paper_url.split('/')[-1]

        # Check for the PDF link.
        pdf_link = next((link.attrib['href'] for link in entry.findall(f'{ARXIV_NAMESPACE}link') 
                         if link.attrib.get('title') == 'pdf'), None)

        papers.append({
            'title': title,
            'summary': summary,
            'authors': authors,
            'arxiv_id': arxiv_id,
            'url': paper_url,
            'pdf_link': pdf_link
        })
    
    # Convert list into a pandas DataFrame.
    df = pd.DataFrame(papers)
    
    # Save the DataFrame to a JSON file.
    with open(json_file_path, 'w', encoding='utf-8') as f:
        json.dump(papers, f, ensure_ascii=False, indent=4)
        print(f'Data saved to {json_file_path} ...')
    
    return df



In [3]:
df = extract_from_arxiv(max_results=20)

Data saved to files/arxiv_dataset.json ...


In [4]:
import json
file_name = 'files/arxiv_dataset.json'

with open(file_name, 'r') as file:
    data = json.load(file)

print(data)

[{'title': 'A Deep Reinforcement Learning Approach for Ramp Metering Based on Traffic Video Data', 'summary': 'Ramp metering that uses traffic signals to regulate vehicle flows from the on-ramps has been widely implemented to improve vehicle mobility of the freeway. Previous studies generally update signal timings in real-time based on predefined traffic measures collected by point detectors, such as traffic volumes and occupancies. Comparing with point detectors, traffic cameras-which have been increasingly deployed on road networks-could cover larger areas and provide more detailed traffic information. In this work, we propose a deep reinforcement learning (DRL) method to explore the potential of traffic video data in improving the efficiency of ramp metering. The proposed method uses traffic video frames as inputs and learns the optimal control strategies directly from the high-dimensional visual inputs. A real-world case study demonstrates that, in comparison with a state-of-the-pr

In [5]:
df =  pd.DataFrame(data)
df.sample(n=5)

,title,summary,authors,arxiv_id,url,pdf_link
9,Compliance Generation for Privacy Documents un...,Most prominent research today addresses compli...,"[David Restrepo Amariles, Aurore ClÃ©ment Trou...",2012.12718v1,http://arxiv.org/abs/2012.12718v1,https://arxiv.org/pdf/2012.12718v1
2,Fuzzy Commitments Offer Insufficient Protectio...,"In this work, we study the protection that fuz...","[Danny Keller, Margarita Osadchy, Orr Dunkelman]",2012.13293v1,http://arxiv.org/abs/2012.13293v1,https://arxiv.org/pdf/2012.13293v1
15,How to define co-occurrence in different domai...,This position paper presents a comparative stu...,[Mathieu Roche],1904.08010v1,http://arxiv.org/abs/1904.08010v1,https://arxiv.org/pdf/1904.08010v1
6,Overview of FPGA deep learning acceleration ba...,"In recent years, deep learning has become more...",[Simin Liu],2012.12634v1,http://arxiv.org/abs/2012.12634v1,https://arxiv.org/pdf/2012.12634v1
14,Devil is in the Edges: Learning Semantic Bound...,We tackle the problem of semantic boundary pre...,"[David Acuna, Amlan Kar, Sanja Fidler]",1904.07934v2,http://arxiv.org/abs/1904.07934v2,https://arxiv.org/pdf/1904.07934v2


### Downloading the Research Papers (PDFs)

In [6]:
import os

def download_pdfs(df, download_folder='files'):
    # Create the folder if does not exist
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)

    pdf_file_names = []

    # Loop through each row fromt he dataframe
    for index, row in df.iterrows():
        pdf_link = row['pdf_link']

        try:
            response = requests.get(pdf_link)
            response.raise_for_status()

            file_name = os.path.join(download_folder, pdf_link.split("/")[-1]) + '.pdf'
            pdf_file_names.append(file_name)

            with open(file_name, 'wb') as file:
                file.write(response.content)

            print(f'PDF downloaded successfully and saved as {file_name}')
        except requests.exceptions.RequestException as e:
            print(f'Failed to download the PDF: {e}')
            pdf_file_names.append(None)
    df['pdf_file_name'] = pdf_file_names

    return df
            

In [7]:
df = download_pdfs(df)

PDF downloaded successfully and saved as files\2012.12104v1.pdf
PDF downloaded successfully and saved as files\2012.13026v1.pdf
PDF downloaded successfully and saved as files\2012.13293v1.pdf
PDF downloaded successfully and saved as files\2012.13315v1.pdf
PDF downloaded successfully and saved as files\2012.13391v2.pdf
PDF downloaded successfully and saved as files\2012.12447v1.pdf
PDF downloaded successfully and saved as files\2012.12634v1.pdf
PDF downloaded successfully and saved as files\2012.11903v1.pdf
PDF downloaded successfully and saved as files\2012.13569v1.pdf
PDF downloaded successfully and saved as files\2012.12718v1.pdf
PDF downloaded successfully and saved as files\2012.13666v1.pdf
PDF downloaded successfully and saved as files\2012.13677v1.pdf
PDF downloaded successfully and saved as files\2012.13779v1.pdf
PDF downloaded successfully and saved as files\2012.14005v1.pdf
PDF downloaded successfully and saved as files\1904.07934v2.pdf
PDF downloaded successfully and saved as

In [8]:
df

,title,summary,authors,arxiv_id,url,pdf_link,pdf_file_name
0,A Deep Reinforcement Learning Approach for Ram...,Ramp metering that uses traffic signals to reg...,"[Bing Liu, Yu Tang, Yuxiong Ji, Yu Shen, Yuchu...",2012.12104v1,http://arxiv.org/abs/2012.12104v1,https://arxiv.org/pdf/2012.12104v1,files\2012.12104v1.pdf
1,Rethink AI-based Power Grid Control: Diving In...,"Recently, deep reinforcement learning (DRL)-ba...","[Xiren Zhou, Siqi Wang, Ruisheng Diao, Desong ...",2012.13026v1,http://arxiv.org/abs/2012.13026v1,https://arxiv.org/pdf/2012.13026v1,files\2012.13026v1.pdf
2,Fuzzy Commitments Offer Insufficient Protectio...,"In this work, we study the protection that fuz...","[Danny Keller, Margarita Osadchy, Orr Dunkelman]",2012.13293v1,http://arxiv.org/abs/2012.13293v1,https://arxiv.org/pdf/2012.13293v1,files\2012.13293v1.pdf
3,Generalization in portfolio-based algorithm se...,Portfolio-based algorithm selection has seen t...,"[Maria-Florina Balcan, Tuomas Sandholm, Ellen ...",2012.13315v1,http://arxiv.org/abs/2012.13315v1,https://arxiv.org/pdf/2012.13315v1,files\2012.13315v1.pdf
4,"I like fish, especially dolphins: Addressing C...",To quantify how well natural language understa...,"[Yixin Nie, Mary Williamson, Mohit Bansal, Dou...",2012.13391v2,http://arxiv.org/abs/2012.13391v2,https://arxiv.org/pdf/2012.13391v2,files\2012.13391v2.pdf
5,Skeleton-based Approaches based on Machine Vis...,"Recently, skeleton-based approaches have achie...","[Jie Li, Binglin Li, Min Gao]",2012.12447v1,http://arxiv.org/abs/2012.12447v1,https://arxiv.org/pdf/2012.12447v1,files\2012.12447v1.pdf
6,Overview of FPGA deep learning acceleration ba...,"In recent years, deep learning has become more...",[Simin Liu],2012.12634v1,http://arxiv.org/abs/2012.12634v1,https://arxiv.org/pdf/2012.12634v1,files\2012.12634v1.pdf
7,Modelling Human Routines: Conceptualising Soci...,Our routines play an important role in a wide ...,"[Rijk Mercuur, Virginia Dignum, Catholijn M. J...",2012.11903v1,http://arxiv.org/abs/2012.11903v1,https://arxiv.org/pdf/2012.11903v1,files\2012.11903v1.pdf
8,Dynamic-K Recommendation with Personalized Dec...,"In this paper, we investigate the recommendati...","[Yan Gao, Jiafeng Guo, Yanyan Lan, Huaming Liao]",2012.13569v1,http://arxiv.org/abs/2012.13569v1,https://arxiv.org/pdf/2012.13569v1,files\2012.13569v1.pdf
9,Compliance Generation for Privacy Documents un...,Most prominent research today addresses compli...,"[David Restrepo Amariles, Aurore ClÃ©ment Trou...",2012.12718v1,http://arxiv.org/abs/2012.12718v1,https://arxiv.org/pdf/2012.12718v1,files\2012.12718v1.pdf


### Loading and Splitting PDF Files into Chunks, Expanding the Dataframe

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_and_chunk_pdf(pdf_file_name, chunk_size=512):
    """
    Loads a PDF file and chunks the content of the file.

    Args:
        pdf_file_name (str) -> path to the PDf file
        chunk_size (int) -> The maximum size of each chunk in characters (default is 512)

    Returns:
        List[Document]: List of document chunks
    """

    print(f'Loading nd splitting into chunks: {pdf_file_name}')
    
    loader = PyPDFLoader(pdf_file_name)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=64)
    chunks = text_splitter.split_documents(data)
    return chunks

In [10]:
def expand_df(df):
    """
    Expands each row in the Dataframe by splitting PDF documents into chunks.

    Args:
        df (pd.DataFrame): DatFrame

    Returns:
        pd.DataFrame: A new DataFrame where each row represents a chunk of the original document, 
                      with additional metadata such as chunk identifiers and relationships to adjacent chunks.
    """

    expanded_rows = [] # List to store exapnded rows with chunk information

    # Loop through each row in the DataFrame
    for idx, row in df.iterrows():
        try:
            chunks = load_and_chunk_pdf(row['pdf_file_name'])
        except Exception as e:
            print(f"Error processing file {row['pdf_file_name']}: {e}")
            continue

        for i, chunk in enumerate(chunks):
            prechunk_id = i-1 if i > 0 else '' # Preceding chunk ID
            post_chunk_id = i+1 if i < len(chunks) - 1 else '' # Following chunk id

            expanded_rows.append({
                'id': f"{row['arxiv_id']}#{i}", # Unique chunk identifier
                'title': row['title'],
                'summary': row['summary'],
                'authors': row['authors'],
                'arxiv_id': row['arxiv_id'],
                'url': row['url'],
                'chunk': chunk.page_content, # Text content of the chunk
                'prechunk_id': '' if i == 0 else f"{row['arxiv_id']}#{prechunk_id}",
                'postchunk_id': '' if i == len(chunks) else f"{row['arxiv_id']}#{post_chunk_id}"
            })
    return pd.DataFrame(expanded_rows)

In [11]:
expanded_df = expand_df(df)

Loading nd splitting into chunks: files\2012.12104v1.pdf
Loading nd splitting into chunks: files\2012.13026v1.pdf
Loading nd splitting into chunks: files\2012.13293v1.pdf
Loading nd splitting into chunks: files\2012.13315v1.pdf
Loading nd splitting into chunks: files\2012.13391v2.pdf
Loading nd splitting into chunks: files\2012.12447v1.pdf
Loading nd splitting into chunks: files\2012.12634v1.pdf
Loading nd splitting into chunks: files\2012.11903v1.pdf
Loading nd splitting into chunks: files\2012.13569v1.pdf
Loading nd splitting into chunks: files\2012.12718v1.pdf
Loading nd splitting into chunks: files\2012.13666v1.pdf
Loading nd splitting into chunks: files\2012.13677v1.pdf
Loading nd splitting into chunks: files\2012.13779v1.pdf
Loading nd splitting into chunks: files\2012.14005v1.pdf
Loading nd splitting into chunks: files\1904.07934v2.pdf
Loading nd splitting into chunks: files\1904.08010v1.pdf
Loading nd splitting into chunks: files\1905.02810v1.pdf
Loading nd splitting into chunk

In [12]:
expanded_df

,id,title,summary,authors,arxiv_id,url,chunk,prechunk_id,postchunk_id
0,2012.12104v1#0,A Deep Reinforcement Learning Approach for Ram...,Ramp metering that uses traffic signals to reg...,"[Bing Liu, Yu Tang, Yuxiong Ji, Yu Shen, Yuchu...",2012.12104v1,http://arxiv.org/abs/2012.12104v1,1 \nA Deep Reinforcement Learning Approach for...,,2012.12104v1#1
1,2012.12104v1#1,A Deep Reinforcement Learning Approach for Ram...,Ramp metering that uses traffic signals to reg...,"[Bing Liu, Yu Tang, Yuxiong Ji, Yu Shen, Yuchu...",2012.12104v1,http://arxiv.org/abs/2012.12104v1,Abstract \nRamp metering that uses traffic sig...,2012.12104v1#0,2012.12104v1#2
2,2012.12104v1#2,A Deep Reinforcement Learning Approach for Ram...,Ramp metering that uses traffic signals to reg...,"[Bing Liu, Yu Tang, Yuxiong Ji, Yu Shen, Yuchu...",2012.12104v1,http://arxiv.org/abs/2012.12104v1,and provide more detailed traffic information....,2012.12104v1#1,2012.12104v1#3
3,2012.12104v1#3,A Deep Reinforcement Learning Approach for Ram...,Ramp metering that uses traffic signals to reg...,"[Bing Liu, Yu Tang, Yuxiong Ji, Yu Shen, Yuchu...",2012.12104v1,http://arxiv.org/abs/2012.12104v1,method results in 1) lower travel times in the...,2012.12104v1#2,2012.12104v1#4
4,2012.12104v1#4,A Deep Reinforcement Learning Approach for Ram...,Ramp metering that uses traffic signals to reg...,"[Bing Liu, Yu Tang, Yuxiong Ji, Yu Shen, Yuchu...",2012.12104v1,http://arxiv.org/abs/2012.12104v1,2 \nIntroduction \nRamp metering uses traffic ...,2012.12104v1#3,2012.12104v1#5
...,...,...,...,...,...,...,...,...,...
2134,1905.02019v1#27,Conditioning LSTM Decoder and Bi-directional A...,Applying neural-networks on Question Answering...,[Heguang Liu],1905.02019v1,http://arxiv.org/abs/1905.02019v1,Prediction: stephen colbert\nReason and Propos...,1905.02019v1#26,1905.02019v1#28
2135,1905.02019v1#28,Conditioning LSTM Decoder and Bi-directional A...,Applying neural-networks on Question Answering...,[Heguang Liu],1905.02019v1,http://arxiv.org/abs/1905.02019v1,5.2.4 Inaccurate boundary\nContext: The Super ...,1905.02019v1#27,1905.02019v1#29
2136,1905.02019v1#29,Conditioning LSTM Decoder and Bi-directional A...,Applying neural-networks on Question Answering...,[Heguang Liu],1905.02019v1,http://arxiv.org/abs/1905.02019v1,information. We can tokenize ”the most” into [...,1905.02019v1#28,1905.02019v1#30
2137,1905.02019v1#30,Conditioning LSTM Decoder and Bi-directional A...,Applying neural-networks on Question Answering...,[Heguang Liu],1905.02019v1,http://arxiv.org/abs/1905.02019v1,characters format feature 2) introducing part-...,1905.02019v1#29,1905.02019v1#31


### Building a knowledge base for the RAG System Using Embedding

In [13]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

True

In [14]:
import os
from getpass import getpass
from semantic_router.encoders import OpenAIEncoder

# Check if 'OPENAI_API_KEY' is set
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key: ')

# Initialize the OpenAIEncoder with a specific model
encoder = OpenAIEncoder(name='text-embedding-3-small')

In [15]:
encoder("hello hallo hota salut")

[[0.0136871337890625,
  0.00644683837890625,
  0.01934814453125,
  0.0301666259765625,
  0.0152130126953125,
  0.002063751220703125,
  -0.0031528472900390625,
  0.050933837890625,
  -0.01546478271484375,
  -0.050018310546875,
  0.01287841796875,
  -0.0015058517456054688,
  -0.07098388671875,
  -0.00457000732421875,
  0.0245208740234375,
  0.01287078857421875,
  -0.024017333984375,
  -0.0032444000244140625,
  0.014251708984375,
  0.041534423828125,
  -0.0208892822265625,
  0.033721923828125,
  0.041168212890625,
  0.039276123046875,
  0.0074005126953125,
  -0.0093841552734375,
  -0.0105133056640625,
  -0.01393890380859375,
  0.0289459228515625,
  -0.01556396484375,
  0.034759521484375,
  -0.03277587890625,
  0.02716064453125,
  -0.02923583984375,
  -0.007694244384765625,
  0.04412841796875,
  -0.049346923828125,
  0.04315185546875,
  -0.0268402099609375,
  0.0228271484375,
  -0.0103302001953125,
  0.00020372867584228516,
  -0.001781463623046875,
  -0.0211181640625,
  0.0316162109375,
  

### Creating a Pinecone Index

In [17]:
from pinecone import Pinecone, ServerlessSpec

# Check if 'PINECONE_API_KEY' is set. Prompt if not.
api_key = os.getenv('PINECONE_API_KEY') or getpass('Pinecone API key:')

In [23]:
# Inititialize the Pinecone client
pc = Pinecone()

# Define the serverless specification for Pinecone (AWS region 'us-east-1')
spec = ServerlessSpec(
    cloud="aws",
    region="us-east-1"
)

In [25]:
import time

# Define the name of the index
index_name = 'research-agent'

# Check if the index exsits
# Create if not
if index_name not in pc.list_indexes().names():
    pc.create_index(
        index_name,
        dimension=1536, # Embedding dimension (1536)
        metric='cosine',
        spec=spec # Cloud provider and region specification
    )

    # Wait until the index is fully initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# Connect to the index
index = pc.Index(index_name)
time.sleep(1)

# View the index statistics
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 27 Mar 2026 18:40:03 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '61',
                                    'x-pinecone-request-latency-ms': '60',
                                    'x-pinecone-response-duration-ms': '63'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}